# Day 6: Data Quality, Validation & Quarantine DLQ Engine

This notebook demonstrates validation using PyDeequ for structural rules and clinical plausibility with DLQ routing.

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit
import pydeequ
from pydeequ.checks import *
from pydeequ.verification import *

os.environ['SPARK_VERSION'] = '3.5'

spark = SparkSession.builder \
    .appName("Day06_DQ_DLQ") \
    .config("spark.jars.packages", pydeequ.deequ_maven_coord) \
    .config("spark.jars.excludes", pydeequ.f2j_maven_coord) \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")


In [ ]:
# Synthetic Data with Intentional Errors
data = [
    ("P1", 32020, "2023-01-01"), # Valid
    (None, 32020, "2023-01-02"), # Invalid: missing person
    ("P3", -5, "2023-01-03")     # Invalid: negative concept_id
]

df = spark.createDataFrame(data, ["person_source_value", "condition_concept_id", "condition_start_date"])

print("Input Data:")
df.show()


In [ ]:
# Define PyDeequ Checks
check = Check(spark, CheckLevel.Error, "OMOP Data Quality Checks") \
    .isComplete("person_source_value") \
    .isNonNegative("condition_concept_id")

# Run Verification
checkResult = VerificationSuite(spark) \
    .onData(df) \
    .addCheck(check) \
    .run()

checkResult_df = VerificationResult.checkResultsAsDataFrame(spark, checkResult)
print("PyDeequ Verification Results:")
checkResult_df.show(truncate=False)


In [ ]:
# Row-Level DLQ Routing
valid_df = df.filter(
    col("person_source_value").isNotNull() & 
    (col("condition_concept_id") >= 0)
)

invalid_df = df.filter(
    col("person_source_value").isNull() | 
    (col("condition_concept_id") < 0)
).withColumn("dq_error_reason", lit("Failed structural/clinical rules"))

print("Valid Records (routed to target):")
valid_df.show()

print("Invalid Records (routed to DLQ):")
invalid_df.show(truncate=False)
